<a href="https://colab.research.google.com/github/ahmedosamasalem/Chest_X-Ray_Detector/blob/main/Ziad's_Final_Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opencv-python tensorflow

In [ ]:
import kagglehub
import cv2
import os
import psutil
import numpy as np
import random
import keras
import tensorflow as tf
from tensorflow.keras import layers
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Path Handling

In [ ]:
# 1. Download & Main Directory
path = kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database")
BASE_DIR = path
print("Path to dataset files:", BASE_DIR)

DATASET_DIR = os.path.join(path ,"COVID-19_Radiography_Dataset")

Using Colab cache for faster access to the 'covid19-radiography-database' dataset.
Path to dataset files: /kaggle/input/covid19-radiography-database


In [ ]:
# 1. Categories
categories = ['COVID', 'Lung_Opacity', 'Normal' , 'Viral Pneumonia']

# 2. Directories
COVID_DIR = os.path.join(DATASET_DIR, "COVID", "images")
NORMAL_DIR = os.path.join(DATASET_DIR, "Normal", "images")
LUNG_OPACITY_DIR = os.path.join(DATASET_DIR, "Lung_Opacity", "images")
VIRAL_PNEUMONIA_DIR = os.path.join(DATASET_DIR, "Viral Pneumonia", "images")

# Preprocessing Pipeline

In [ ]:
IMAGE_SIZE = (224,224)
BATCH_SIZE = 64

def load_images_from_folder(folder_path, label):
    images = []
    labels = []

    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    for filename in image_files:
        file_path = os.path.join(folder_path, filename)
        image = cv2.imread(file_path)                  # reads as BGR
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # convert to RGB
        image = cv2.resize(image, IMAGE_SIZE)  # resize
        images.append(image)
        labels.append(label)

    return images, labels

In [ ]:
print("Loading training covid images...")
covid_images, covid_labels = load_images_from_folder(COVID_DIR, 0)
print(f"Loaded {len(covid_images)} training covid images.")

print("Loading training normal images...")
normal_images, normal_labels = load_images_from_folder(NORMAL_DIR, 1)
print(f"Loaded {len(normal_images)} training normal images.")

print("Loading training lung opacity images...")
lung_opacity_images, lung_opacity_labels = load_images_from_folder(LUNG_OPACITY_DIR, 2)
print(f"Loaded {len(lung_opacity_images)} training lung opacity images.")

print("Loading training viral pneumonia images...")
viral_pneumonia_images, viral_pneumonia_labels = load_images_from_folder(VIRAL_PNEUMONIA_DIR, 3)
print(f"Loaded {len(viral_pneumonia_images)} training viral pneumonia images.")

Loading training covid images...
Loaded 3616 training covid images.
Loading training normal images...
Loaded 10192 training normal images.
Loading training lung opacity images...
Loaded 6012 training lung opacity images.
Loading training viral pneumonia images...
Loaded 1345 training viral pneumonia images.


#Train Test Split

In [ ]:
X = np.array(covid_images +normal_images +lung_opacity_images +viral_pneumonia_images)
y = np.array(covid_labels +normal_labels +lung_opacity_labels +viral_pneumonia_labels)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Train samples: {len(X_train)}, Validation samples: {len(X_val)}")

Train samples: 16932, Validation samples: 4233


In [ ]:
import tensorflow as tf
from keras.applications.mobilenet_v2 import preprocess_input

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))

def preprocess_batch(image, label):
    image = preprocess_input(tf.cast(image, tf.float32))
    return image, label

train_dataset = train_dataset.map(preprocess_batch, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(1000).batch(64).prefetch(tf.data.AUTOTUNE)

val_dataset = val_dataset.map(preprocess_batch, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(64).prefetch(tf.data.AUTOTUNE)

# The Model

In [ ]:
# (Data Augmentation)
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3),include_top=False,weights="imagenet")

base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(4, activation="softmax")
])


In [ ]:
model.compile(optimizer="Adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])


In [ ]:
# 3.EarlyStopping
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    verbose=1,
    mode="auto",
    restore_best_weights=True,
    start_from_epoch=3
)

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks=[early_stopping]
)

In [ ]:
model.summary()

In [ ]:
model.save("model_2.keras")
print("Model saved successfully!")
files.download("model_2.keras")
print("Model downloaded successfully!")

In [ ]:
y_train_pred = model.predict(X_train, batch_size=32)
y_train_pred = np.argmax(y_train_pred, axis=1)
print(classification_report(y_train, y_train_pred))
del y_train_pred

y_test_pred = model.predict(X_test, batch_size=32)
y_test_pred = np.argmax(y_test_pred, axis=1)
print(classification_report(y_test, y_test_pred))
